# UABR-SLM · Notebook 03 — Pretrain the 40M model (2× T4, DDP)

**Phase 03 of 11** (PLAN §4, amendment A6). GPU **T4 ×2** · ~3–4 h expected (measured below) · output ~0.6 GB.

**Inputs (Add Input → Your Work):** notebook **01** output (`train.bin`, `val.bin`, tokenizer, exposure table).
To **resume** after an interruption, also attach this notebook's own previous output: the checkpoint is picked up automatically.
**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** (to clone the code) · **Save & Run All**.

**Model (team decisions):** decoder-only LLaMA-style, 8 layers × d_model 512, 8 heads (head dim 64), **SwiGLU d_ff 2048**,
RoPE, pre-norm RMSNorm, no biases, tied embeddings, vocab 12k, context 512 → **39,707,136 parameters**.

**Training recipe (defaults; change in the config cell before running):**

| setting | value | why |
|---|---|---|
| data order | **one exact pass**, 512-token blocks, fixed permutation, split across GPUs | every exposure copy is seen exactly once |
| global batch | 256 × 512 = 131k tokens/step (32 per GPU × 4 accumulation × 2 GPUs) | ~7.9k optimizer steps |
| optimizer | AdamW β = (0.9, 0.95), wd 0.1 (not on norms), grad clip 1.0 | standard small-LM setup |
| learning rate | 1e-3 peak, 500 warm-up steps, cosine to 1e-4 | standard for ~40M models |
| precision | fp16 autocast + GradScaler (T4 has no bf16) | |
| checkpoints | every 30 min + at the end; stop safely after 11 h | resume across Kaggle sessions |

## 1 · Get the project code

In [ ]:
import os, sys, glob, json, subprocess, platform, time, shutil
ON_KAGGLE = os.path.exists("/kaggle/working")

REPO_URL = "https://github.com/Prajwal-13-dev/uabr_slm.git"
REPO_REF = "main"        # pin to a commit hash for final, reportable runs

def find_code_root():
    if os.environ.get("UABR_CODE"):
        return os.path.abspath(os.environ["UABR_CODE"])
    for base in glob.glob("/kaggle/input/*"):
        for root, dirs, _ in os.walk(base):
            if os.path.isdir(os.path.join(root, "src", "uabr")):
                return root
            if root.count(os.sep) - base.count(os.sep) >= 4:
                dirs[:] = []
    dest = "/tmp/uabr_slm"
    if not os.path.isdir(dest):
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = url.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", url, dest], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", REPO_REF], check=True)
    return dest

CODE_ROOT = find_code_root()
SRC = os.path.join(CODE_ROOT, "src")
sys.path.insert(0, SRC)
CODE_COMMIT = subprocess.run(["git", "-C", CODE_ROOT, "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "unknown"

def find_input(pattern, env=None):
    """Locate a file from an attached notebook output (or an env override for local tests)."""
    if env and os.environ.get(env):
        return os.environ[env]
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input - attach the required notebook output (Add Input → Your Work)"
    return hits[0]

T0 = time.time()
def stage(msg): print(f"[{(time.time()-T0)/60:6.1f} min] {msg}", flush=True)
print("code:", CODE_ROOT, "| commit:", CODE_COMMIT[:10])

import torch
from uabr.model.llama import LlamaLM, ModelConfig, expected_param_count
from uabr.data import tokenizer as T

## 2 · Inputs, GPUs, configuration

In [ ]:
import numpy as np, pandas as pd
OUT = "/kaggle/working/03_pretrain" if ON_KAGGLE else os.path.abspath("outputs/03_pretrain")
os.makedirs(OUT, exist_ok=True)
TRAIN_BIN = find_input("01_corpus/pretrain/train.bin", "TRAIN_BIN")
VAL_BIN = find_input("01_corpus/pretrain/val.bin", "VAL_BIN")
TOK_PATH = find_input("01_corpus/tokenizer/tokenizer.json", "TOKENIZER")

# resume: copy a checkpoint from an attached previous run of this notebook
if not os.path.exists(f"{OUT}/ckpt_latest.pt"):
    prev = sorted(glob.glob("/kaggle/input/**/03_pretrain/ckpt_latest.pt", recursive=True))
    if prev:
        for f in glob.glob(os.path.join(os.path.dirname(prev[0]), "*")):
            if os.path.isfile(f): shutil.copy(f, OUT)
        print("resuming from attached checkpoint:", prev[0])

N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i); print(f"GPU {i}: {p.name}, {p.total_memory/1e9:.0f} GB, sm{p.major}{p.minor}")
NPROC = max(1, N_GPU)
train_tokens = os.path.getsize(TRAIN_BIN) // 2
print(f"train.bin {train_tokens/1e9:.3f}B tokens | val.bin {os.path.getsize(VAL_BIN)//2/1e6:.1f}M tokens | processes {NPROC}")

MODEL = dict(vocab_size=12000, n_layers=8, d_model=512, n_heads=8, d_ff=2048)
RUN = dict(
    train_bin=TRAIN_BIN, val_bin=VAL_BIN, out_dir=OUT, model=MODEL, seq_len=512,
    global_batch_seqs=256, micro_batch_seqs=32, lr=1e-3, min_lr=1e-4, warmup_steps=500,
    weight_decay=0.1, beta1=0.9, beta2=0.95, grad_clip=1.0, precision="auto", compile=False,
    seed=1234, data_seed=4321, eval_every=500, eval_tokens=2_000_000, log_every=20,
    ckpt_minutes=30, max_hours=11.0, max_steps=int(os.environ.get("MAX_STEPS", 0)),
)
for k in ["global_batch_seqs", "micro_batch_seqs", "eval_tokens", "eval_every", "log_every", "warmup_steps"]:
    if os.environ.get(k.upper()): RUN[k] = int(os.environ[k.upper()])
if os.environ.get("MODEL_JSON"): MODEL.update(json.loads(os.environ["MODEL_JSON"])); RUN["model"] = MODEL
json.dump(RUN, open(f"{OUT}/run_config.json", "w"), indent=2)
RUN

## 3 · Model check: exact parameter count (A6)

In [ ]:
mcfg = ModelConfig(**{**ModelConfig().to_dict(), **MODEL})
m = LlamaLM(mcfg)
N_PARAMS = m.num_params()
print(f"d_ff = {mcfg.d_ff} | parameters = {N_PARAMS:,} (analytic {expected_param_count(mcfg):,})")
print(f"  embedding (tied) {mcfg.vocab_size*mcfg.d_model:,} | per block {(N_PARAMS - mcfg.vocab_size*mcfg.d_model - mcfg.d_model)//mcfg.n_layers:,}")
del m

## 4 · Train
Runs `torchrun` on all GPUs; the log streams below and is saved to `train_stdout.log`. If the session dies, re-run with this notebook's output attached; training resumes from the last checkpoint.

In [ ]:
env = {**os.environ, "PYTHONPATH": SRC + os.pathsep + os.environ.get("PYTHONPATH", ""),
       "NCCL_P2P_DISABLE": os.environ.get("NCCL_P2P_DISABLE", "1"),   # safe default on PCIe T4s
       "OMP_NUM_THREADS": "2"}
cmd = [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={NPROC}",
       "-m", "uabr.train.pretrain", "--config", f"{OUT}/run_config.json"]
summary_path = f"{OUT}/summary.json"
already = os.path.exists(summary_path) and json.load(open(summary_path)).get("completed") and not RUN["max_steps"]
if already:
    print("training already completed in attached output - skipping")
else:
    with open(f"{OUT}/train_stdout.log", "a") as logf:
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end=""); logf.write(line)
        rc = p.wait()
    assert rc == 0, f"training exited with code {rc} - see train_stdout.log"
SUMMARY = json.load(open(summary_path))
{k: SUMMARY[k] for k in ["params", "world", "precision", "steps_done", "total_steps", "completed",
                         "tokens_trained", "dropped_tail_tokens", "throughput_tok_per_s_median", "wall_minutes_this_session"]}

## 5 · Loss curves

In [ ]:
import matplotlib.pyplot as plt
log = pd.read_csv(f"{OUT}/train_log.csv")
tr, va = log.dropna(subset=["train_loss"]), log.dropna(subset=["val_loss"])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(tr["tokens"] / 1e9, tr["train_loss"], lw=0.8, label="train")
ax[0].plot(va["tokens"] / 1e9, va["val_loss"], "o-", label="val")
ax[0].set(xlabel="tokens (B)", ylabel="loss (nats)", title="Pretraining loss", ylim=(None, min(12, va["val_loss"].max() + 0.5))); ax[0].legend()
ax[1].plot(tr["step"], tr["tok_per_s"] / 1e3, lw=0.8); ax[1].set(xlabel="step", ylabel="k tokens / s", title="Throughput (all GPUs)")
plt.tight_layout(); plt.savefig(f"{OUT}/loss_curves.png", dpi=120); plt.show()
print(va[["step", "val_loss"]].to_string(index=False))

## 6 · Qualitative samples (greedy) — High-exposure vs Control subjects

In [ ]:
tok = T.load(TOK_PATH)
dev = "cuda" if torch.cuda.is_available() else "cpu"
ck = f"{OUT}/model_final.pt" if os.path.exists(f"{OUT}/model_final.pt") else f"{OUT}/ckpt_latest.pt"
state = torch.load(ck, map_location=dev, weights_only=False)
model = LlamaLM(ModelConfig(**state["model_cfg"])).to(dev).eval()
model.load_state_dict(state["model"])
try:
    expo = pd.read_csv(find_input("01_corpus/exposure/exposure_assignment.csv", "EXPOSURE"))
    picks = pd.concat([expo[expo["exposure_group"] == g].sample(3, random_state=1) for g in ["high", "control"]])
    prompts = [(g, t.split(" (")[0]) for g, t in zip(picks["exposure_group"], picks["title_n"])]
except AssertionError:
    prompts = []
prompts += [("generic", "The city of"), ("generic", "In 1998, the band")]
for g, ptxt in prompts:
    ids = torch.tensor([[T.BOS_ID] + tok.encode(ptxt, add_special_tokens=False).ids], device=dev)
    with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):
        out = model.generate(ids, 30, eos_id=T.EOS_ID)
    print(f"[{g:7}] {tok.decode(out[0, 1:].tolist())!r}")

## 7 · Gate 03

In [ ]:
checks = []
def add(name, ok, detail, severity="fail"):
    checks.append({"name": name, "passed": bool(ok), "detail": detail, "severity": severity})
add("training_completed", SUMMARY["completed"], f"{SUMMARY['steps_done']}/{SUMMARY['total_steps']} steps (if not: re-run with this output attached)")
add("param_count_39_7M", SUMMARY["params"] == expected_param_count(mcfg), f"{SUMMARY['params']:,}")
v0, vN = float(va["val_loss"].iloc[0]), float(va["val_loss"].iloc[-1])
add("val_loss_finite_and_improved", np.isfinite(vN) and vN < v0 - 1.0, f"val loss {v0:.3f} -> {vN:.3f}")
add("model_final_saved", os.path.exists(f"{OUT}/model_final.pt"), "model_final.pt")
tp = SUMMARY.get("throughput_tok_per_s_median")
add("throughput_recorded", tp is not None, f"{(tp or 0)/1e3:.1f}k tokens/s on {SUMMARY['world']} GPU(s)", "warn")

fails = [c for c in checks if not c["passed"] and c["severity"] == "fail"]
warns = [c for c in checks if not c["passed"] and c["severity"] == "warn"]
status = "FAIL" if fails else ("PASS_WITH_WARNINGS" if warns else "PASS")
for c in checks:
    print(f"[{'PASS' if c['passed'] else c['severity'].upper():4}] {c['name']:<38} {c['detail']}")

json.dump({"gate": "03_pretrain", "status": status, "checks": checks, "final_val_loss": vN,
           "code_commit": CODE_COMMIT}, open(f"{OUT}/gate_report.json", "w"), indent=2, default=str)
print("\nGATE 03:", status)
assert not fails, "Pretraining gate failed."

## Next
- **Notebook 04 (G1 pilot)** needs this `model_final.pt` + notebook 00/01 outputs. Before it is built, the team must decide the closed-book prompt format (open item in PLAN §15).
- Delete `ckpt_latest.pt` from the output only after `model_final.pt` is confirmed (it also holds optimizer state, ~0.5 GB).